# Lunar Road Demo: SysML v2 model

Run all cells in order. Each model cell should report the package it created; any error names the line.

## 00_Metadata

In [ ]:
package LRD_Metadata {
    doc /*
     * Project metadata used on every requirement: where it comes from in the specs, and how
     * it relates to the original wording (kept, reworded, split or new).
     */

    private import ScalarValues::String;

    metadata def SpecSource {
        attribute reference : String;
        attribute basis : String;
    }
}


## 01_Structure

In [ ]:
package LRD_Structure {
    doc /*
     * Lunar Road Demo MVP: logical architecture of the simulation system.
     * Source: "Lunar Road Demo MVP: System Boundary and Simulation Conditions".
     * Sourced values are bound with '='. Assumed values use 'default' so they
     * stay configurable, as the spec requires.
     */

    private import ScalarValues::*;
    private import ISQ::*;
    private import SI::*;

    // =====================================================================
    // External context: things the system uses but cannot change
    // =====================================================================

    part def LunarTerrain {
        doc /* NASA enhanced 5 m/pixel model for Connecting Ridge, LOLA Site01 as fallback. */
        port terrainOut : TerrainPort;
    }

    part def SunAndEarthEphemeris {
        doc /* SPICE kernels or a table giving Sun and Earth azimuth and elevation. */
        port ephemerisOut : EphemerisPort;
    }

    part def DemoViewer {
        doc /* Person running the comparison and changing the road assumptions. */
    }

    part def DemoPresentation {
        doc /* Shared results, the viewer display and anything shown with the tabletop rover.
             * Subject of the presentation requirements (caveats and framing). */
    }

    // =====================================================================
    // Items carried on the interfaces (IF-01 to IF-07)
    // =====================================================================

    item def TerrainData {
        doc /* IF-01: height (m), slope and wider horizon terrain. GeoTIFF on the LOLA polar stereographic grid.
             * The 5 m cell size applies to the 2 km patch; the wider horizon terrain is lower resolution. */
        attribute patchCellSize : LengthValue = 5 [m];
    }

    item def EphemerisData {
        doc /* IF-02: time, Sun and Earth azimuth and elevation (degrees). */
        attribute time : DurationValue;
        attribute sunAzimuth : AngularMeasureValue;
        attribute sunElevation : AngularMeasureValue;
        attribute earthAzimuth : AngularMeasureValue;
        attribute earthElevation : AngularMeasureValue;
    }

    item def EnvironmentMaps {
        doc /* IF-03: slope, lit flag, Earth flag and road flag by cell and time. Gridded arrays. */
        attribute slope : AngularMeasureValue;
        attribute isLit : Boolean;
        attribute earthInView : Boolean;
        attribute isRoad : Boolean;
    }

    item def Command {
        doc /* IF-04: planner to rover, through the delay. JSON message. */
        attribute commandId : String;
        attribute targetX : LengthValue;
        attribute targetY : LengthValue;
        attribute heading : AngularMeasureValue;
        attribute mode : String;
    }

    item def RoverStatus {
        doc /* IF-05: rover to planner, through the delay. JSON message. */
        attribute time : DurationValue;
        attribute x : LengthValue;
        attribute y : LengthValue;
        attribute heading : AngularMeasureValue;
        attribute stateOfCharge : Real;
        attribute mode : String;
        attribute commandStatus : String;
    }

    item def RunRecord {
        doc /* IF-06: route, state, solar power, load, events and results. Parquet or CSV. */
        attribute time : DurationValue;
        attribute x : LengthValue;
        attribute y : LengthValue;
        attribute stateOfCharge : Real;
        attribute solarPower : PowerValue;
        attribute load : PowerValue;
        attribute eventText : String;
    }

    item def ScenarioSettings {
        doc /* IF-07: road on or off, preset, start, goal, start time, seed and parameters. Versioned YAML. */
        attribute roadOn : Boolean;
        attribute preset : String;
        attribute startX : LengthValue;
        attribute startY : LengthValue;
        attribute goalX : LengthValue;
        attribute goalY : LengthValue;
        attribute startTime : String;
        attribute seed : Integer;
    }

    // =====================================================================
    // Ports
    // =====================================================================

    port def TerrainPort      { out item terrain : TerrainData; }
    port def EphemerisPort    { out item ephemeris : EphemerisData; }
    port def EnvMapsPort      { out item maps : EnvironmentMaps; }
    port def CommandPort      { out item command : Command; }
    port def StatusPort       { out item status : RoverStatus; }
    port def RunRecordPort    { out item record : RunRecord; }
    port def ScenarioPort     { out item settings : ScenarioSettings; }

    // =====================================================================
    // Interface definitions, one per IF in the spec
    // =====================================================================

    interface def 'IF-01 TerrainInterface'   { end source : TerrainPort;   end sink : ~TerrainPort; }
    interface def 'IF-02 EphemerisInterface' { end source : EphemerisPort; end sink : ~EphemerisPort; }
    interface def 'IF-03 EnvMapsInterface'   { end source : EnvMapsPort;   end sink : ~EnvMapsPort; }
    interface def 'IF-04 CommandInterface'   { end source : CommandPort;   end sink : ~CommandPort; }
    interface def 'IF-05 StatusInterface'    { end source : StatusPort;    end sink : ~StatusPort; }
    interface def 'IF-06 RunRecordInterface' { end source : RunRecordPort; end sink : ~RunRecordPort; }
    interface def 'IF-07 ScenarioInterface'  { end source : ScenarioPort;  end sink : ~ScenarioPort; }

    // =====================================================================
    // Logical elements (the five simulation models plus display and config)
    // =====================================================================

    part def EnvironmentModel {
        doc /* Model 3: Sun and Earth visibility from terrain horizons; road cells from a fixed centreline. */
        attribute mapArea : LengthValue default 2000 [m];
        attribute roadWidth : LengthValue default 10 [m];
        attribute environmentStep : DurationValue default 15 [min];
        attribute earthVisibilityHeight : LengthValue default 2 [m];
        attribute sunlightHeight : LengthValue default 1 [m];
        port terrainIn : ~TerrainPort;
        port ephemerisIn : ~EphemerisPort;
        port mapsOut : EnvMapsPort;
        port settingsIn : ~ScenarioPort;
    }

    part def MobilityModel {
        doc /* Model 1: speed made good on flat ground, reduced by slip on slopes. */
        attribute nominalSpeed : SpeedValue = 10 [cm/s];
        attribute topSpeed : SpeedValue = 20 [cm/s];
        attribute offRoadSpeedMadeGood : SpeedValue = 0.8 [cm/s];
        attribute slopeLimit : AngularMeasureValue = 15 ['°'];
        attribute maxHeadingOffset : AngularMeasureValue default 50 ['°'];
        attribute offRoadRollingResistance : Real default 0.2;
        attribute movingFraction : Real;   // constant for each preset (ASM-P1)
        attribute currentSlope : AngularMeasureValue;
        attribute slip : Real;            // off-road: 2% at 0, 5% at 5, 10% at 10, 20% at 15 degrees (assumed)
    }

    part def PowerModel {
        doc /* Model 2: battery state updated each step as solar input minus load. */
        attribute batteryCapacity : EnergyValue = 5420 [W*h];   // at start of life, 0 C
        attribute solarPerPanel : PowerValue = 320 [W];
        attribute solarAtCorner : PowerValue = 450 [W];
        attribute batteryReserveFraction : Real default 0.2;
        attribute awakeLoad : PowerValue default 250 [W];
        attribute drivetrainEfficiency : Real default 0.5;
        attribute stateOfCharge : Real;
    }

    part def RoverModel {
        doc /* One VIPER-class rover. Mass and footprint from NASA 2024 slides (source A). */
        attribute rovingMass : MassValue = 447 [kg];
        attribute footprint : LengthValue = 1.7 [m];
        part mobility : MobilityModel;
        part power : PowerModel;
        port mapsIn : ~EnvMapsPort;
        port commandIn : ~CommandPort;
        port statusOut : StatusPort;
        port recordOut : RunRecordPort;
        port settingsIn : ~ScenarioPort;
    }

    part def CommandDelayLink {
        doc /* Model 4: each message waits half of a sampled 6 to 10 s round trip; optional drops. */
        attribute minRoundTrip : DurationValue = 6 [s];
        attribute maxRoundTrip : DurationValue = 10 [s];
        attribute statusPeriod : DurationValue default 60 [s];
        attribute ackTimeout : DurationValue default 15 [s];   // REQ-C2.3 (TBR); spec said 10 s
        attribute packetDropRate : Real default 0.0;
        port mapsIn : ~EnvMapsPort;
        port commandIn : ~CommandPort;
        port commandOut : CommandPort;
        port statusIn : ~StatusPort;
        port statusOut : StatusPort;
        port settingsIn : ~ScenarioPort;
    }

    part def Planner {
        doc /* Model 5: the same time-aware planner runs in both cases, so any difference comes from the road. */
        attribute commandCycle : DurationValue = 5 [min];
        attribute minWaypointSpacing : LengthValue = 4 [m];   // off-road, sourced
        attribute maxWaypointSpacing : LengthValue = 8 [m];   // off-road, sourced; road limit set by preset
        attribute energyPenalty : Real default 0.001;   // hours per Wh drawn
        attribute maxPlanTime : DurationValue default 5 [min];
        attribute maxPlannedSlope : AngularMeasureValue;
        port mapsIn : ~EnvMapsPort;
        port commandOut : CommandPort;
        port statusIn : ~StatusPort;
        port recordOut : RunRecordPort;
        port settingsIn : ~ScenarioPort;
    }

    part def DisplayAndLog {
        doc /* Display over the stored run set, plus the run log. Product form not yet decided (REQ-MVP10). */
        attribute storedRuns : Integer = 24;
        port plannerRecordIn : ~RunRecordPort;
        port roverRecordIn : ~RunRecordPort;
        port settingsIn : ~ScenarioPort;
    }

    part def ScenarioConfig {
        doc /* Versioned YAML scenario: start, goal, start time, seed, road preset. */
        attribute scenarioLength : DurationValue default 96 [h];
        attribute startToGoal : LengthValue default 1500 [m];
        attribute initialCharge : Real default 1.0;
        port settingsOut : ScenarioPort;
    }

    // =====================================================================
    // The system of interest, with every interface connected
    // =====================================================================

    part def LunarRoadDemoSystem {
        doc /* Simulates one rover, its route and the assumed effects of a road. */
        part environment : EnvironmentModel;
        part planner : Planner;
        part commLink : CommandDelayLink;
        part rover : RoverModel;
        part display : DisplayAndLog;
        part scenario : ScenarioConfig;

        // Boundary ports: external inputs delegated to the environment model
        port terrainIn : ~TerrainPort;
        port ephemerisIn : ~EphemerisPort;
        bind terrainIn = environment.terrainIn;
        bind ephemerisIn = environment.ephemerisIn;

        // IF-03: environment to planner, link and rover
        interface envToPlanner : 'IF-03 EnvMapsInterface' connect
            source ::> environment.mapsOut to sink ::> planner.mapsIn;
        interface envToLink : 'IF-03 EnvMapsInterface' connect
            source ::> environment.mapsOut to sink ::> commLink.mapsIn;
        interface envToRover : 'IF-03 EnvMapsInterface' connect
            source ::> environment.mapsOut to sink ::> rover.mapsIn;

        // IF-04: planner to rover, through the delay
        interface cmdToLink : 'IF-04 CommandInterface' connect
            source ::> planner.commandOut to sink ::> commLink.commandIn;
        interface cmdToRover : 'IF-04 CommandInterface' connect
            source ::> commLink.commandOut to sink ::> rover.commandIn;

        // IF-05: rover to planner, through the delay
        interface statusToLink : 'IF-05 StatusInterface' connect
            source ::> rover.statusOut to sink ::> commLink.statusIn;
        interface statusToPlanner : 'IF-05 StatusInterface' connect
            source ::> commLink.statusOut to sink ::> planner.statusIn;

        // IF-06: planner and rover to display and log
        interface plannerToLog : 'IF-06 RunRecordInterface' connect
            source ::> planner.recordOut to sink ::> display.plannerRecordIn;
        interface roverToLog : 'IF-06 RunRecordInterface' connect
            source ::> rover.recordOut to sink ::> display.roverRecordIn;

        // IF-07: scenario settings to all blocks
        interface settingsToEnv : 'IF-07 ScenarioInterface' connect
            source ::> scenario.settingsOut to sink ::> environment.settingsIn;
        interface settingsToPlanner : 'IF-07 ScenarioInterface' connect
            source ::> scenario.settingsOut to sink ::> planner.settingsIn;
        interface settingsToRover : 'IF-07 ScenarioInterface' connect
            source ::> scenario.settingsOut to sink ::> rover.settingsIn;
        interface settingsToLink : 'IF-07 ScenarioInterface' connect
            source ::> scenario.settingsOut to sink ::> commLink.settingsIn;
        interface settingsToDisplay : 'IF-07 ScenarioInterface' connect
            source ::> scenario.settingsOut to sink ::> display.settingsIn;
    }

    // =====================================================================
    // Context: the system with its external inputs (IF-01, IF-02)
    // =====================================================================

    part demoContext {
        part terrain : LunarTerrain;
        part ephemeris : SunAndEarthEphemeris;
        part viewer : DemoViewer;
        part sim : LunarRoadDemoSystem;

        interface terrainLink : 'IF-01 TerrainInterface' connect
            source ::> terrain.terrainOut to sink ::> sim.terrainIn;
        interface ephemerisLink : 'IF-02 EphemerisInterface' connect
            source ::> ephemeris.ephemerisOut to sink ::> sim.ephemerisIn;
    }
}


## 02_Context_UseCases

In [ ]:
package LRD_Context {
    doc /*
     * Mission context and use cases for the Lunar Road Demo MVP.
     * The demo is complete when a viewer can run both cases, change the road
     * assumptions and see how the result changes.
     */

    public import LRD_Structure::*;

    use case def 'Compare Trip With And Without Road' {
        subject sim : LunarRoadDemoSystem;
        actor viewer : DemoViewer;
        actor terrain : LunarTerrain;
        actor ephemeris : SunAndEarthEphemeris;
        objective {
            doc /* Show the difference a road makes to the same VIPER-class trip in time,
                 * energy, risk and reliability, on real south polar terrain. */
        }
    }

    use case def 'Change Road Assumptions' {
        subject sim : LunarRoadDemoSystem;
        actor viewer : DemoViewer;
        objective {
            doc /* Switch road on or off, pick the half, expected or high road preset, and pick
                 * a 150 W, 250 W or 350 W awake load, then see results update from the stored run set. */
        }
    }

    use case def 'Inspect Run Results' {
        subject sim : LunarRoadDemoSystem;
        actor viewer : DemoViewer;
        objective {
            doc /* Read arrival time, distance, energy used, minimum state of charge, time in shadow,
                 * waiting time, peak slip and pass or fail for each run. */
        }
    }

    use case 'run the demo' : 'Compare Trip With And Without Road' {
        first start;
        then include use case 'change assumptions' : 'Change Road Assumptions';
        then include use case 'inspect results' : 'Inspect Run Results';
        then done;
    }
}


## 03_Requirements_Sim

In [ ]:
package LRD_Requirements_Sim {
    doc /*
     * Simulation and presentation requirements, rewritten to INCOSE rules (one 'shall' each,
     * verifiable, with rationale and source). REQ-M, P, E, C and R keep the spec's IDs; a
     * decimal suffix (REQ-C2.3) marks one part of a spec requirement that was split. REQ-MVP
     * and REQ-IF IDs were assigned in this model. Modelling assumptions (ASM-*) are recorded
     * as assume constraints, not requirements. StatusInfo tbr marks a proposed value to review.
     */

    private import ScalarValues::*;
    private import ISQ::*;
    private import SI::*;
    private import ModelingMetadata::*;
    private import LRD_Metadata::*;
    public import LRD_Structure::*;

    requirement def SimRequirement {
        doc /* Placed on the whole demo system. */
        subject sim : LunarRoadDemoSystem;
    }

    requirement def PresentationRequirement {
        doc /* Placed on how results and the rover are presented, not on the simulation. */
        subject presentation : DemoPresentation;
    }


    // ---------------------------------------------------------------------
    // Demo-level requirements
    // ---------------------------------------------------------------------

    requirement <'REQ-MVP1.1'> roadComparison : SimRequirement {
        doc /* The system shall run the same VIPER-class rover trip on real lunar south polar terrain
             * twice, once with the road off and once with the road on. */
        @Rationale { text = "This comparison is the purpose of the demo."; }
        @SpecSource { reference = "MVP spec, Purpose"; basis = "Split from the Purpose statement"; }
    }

    requirement <'REQ-MVP1.2'> reportedResults : SimRequirement {
        doc /* For each run, the system shall report arrival time, distance, energy used, minimum
             * state of charge, time in shadow, waiting time, peak slip, and pass or fail. */
        @Rationale { text = "These are the outputs a viewer compares between the two runs."; }
        @SpecSource { reference = "MVP spec, Purpose; Demo scenarios (reported results)"; basis = "Split from the Purpose statement"; }
    }

    requirement <'REQ-MVP1.3'> riskAndReliability : SimRequirement {
        doc /* The system shall report risk for each run as the minimum state-of-charge margin above
             * reserve, the peak slip and the time in shadow. */
        @Rationale { text = "The Purpose statement names risk but does not say how it is measured; this defines it from outputs the spec already lists."; }
        @SpecSource { reference = "MVP spec, Purpose"; basis = "New definition"; }
        @StatusInfo { status = StatusKind::tbr; }
    }

    requirement <'REQ-MVP1.4'> reliabilityMeasure : SimRequirement {
        doc /* The system shall report reliability for each run as the number of dropped messages and
             * the number of resent commands. */
        @Rationale { text = "The Purpose statement names reliability but does not say how it is measured. Both counts are zero in the stored run set, which uses 0% packet drops."; }
        @SpecSource { reference = "MVP spec, Purpose"; basis = "New definition"; }
        @StatusInfo { status = StatusKind::tbr; }
    }

    requirement <'REQ-MVP1.5'> comparisonDifference : SimRequirement {
        doc /* For each comparison, the system shall report the road-on minus road-off difference of
             * every reported result. */
        @Rationale { text = "The Purpose is to show the difference the road makes, not only each run's values."; }
        @SpecSource { reference = "MVP spec, Purpose"; basis = "Split from the Purpose statement"; }
    }

    requirement <'REQ-MVP2'> reportEitherWay : PresentationRequirement {
        doc /* Every presentation of the results shall report them whichever way they turn out,
             * including when the road makes little difference. */
        @Rationale { text = "A demo that only shows favourable results would mislead."; }
        @SpecSource { reference = "MVP spec, Demo scenarios"; basis = "Reworded from the Demo scenarios statement"; }
    }

    requirement <'REQ-MVP3.1'> caveatSimplifiedModel : PresentationRequirement {
        doc /* Every shared result shall state that it comes from a simplified model using assumed
             * rover loads, slip and road performance. */
        @Rationale { text = "Viewers must not read the results as predictions of real road performance."; }
        @SpecSource { reference = "MVP spec, Required caveat"; basis = "Split from the required caveat"; }
    }

    requirement <'REQ-MVP3.2'> caveatAutonomy : PresentationRequirement {
        doc /* Every shared result shall state that VIPER is driven by a human team, while this demo
             * plans automatically. */
        @Rationale { text = "The automatic planner is a simplification of how VIPER is really operated."; }
        @SpecSource { reference = "MVP spec, Required caveat"; basis = "Split from the required caveat"; }
    }

    requirement <'REQ-MVP4'> assumedValuesConfigurable : SimRequirement {
        doc /* The system shall expose every value marked Assumed in this spec as a setting in the
             * scenario file (IF-07). */
        @Rationale { text = "Assumed values are placeholders that must change as test data arrives."; }
        @SpecSource { reference = "MVP spec, Parameters"; basis = "Reworded from 'every assumed value shall be configurable'"; }
    }

    requirement <'REQ-MVP5'> fixedScenario : SimRequirement {
        doc /* Both runs of a comparison shall use the same start, goal, start time and terrain map. */
        @Rationale { text = "The road must be the only difference between the runs."; }
        @SpecSource { reference = "MVP spec, Environment conditions"; basis = "Reworded from 'each comparison shall use one fixed start, goal, start time and terrain map'"; }
    }

    requirement <'REQ-MVP6'> widerTerrainForShadows : SimRequirement {
        doc /* The shadow calculation shall include a lower-resolution terrain map extending beyond
             * the 2 km patch, with its extent to be decided. */
        @Rationale { text = "Distant ridges cast shadows into the patch. The wider map is still an open decision."; }
        @SpecSource { reference = "MVP spec, Environment conditions (source E)"; basis = "Reworded from the Environment conditions statement"; }
        @StatusInfo { status = StatusKind::tbr; }
    }

    requirement <'REQ-MVP7.1'> demoSite : SimRequirement {
        doc /* The demo shall use the Connecting Ridge site, or Connecting Ridge Extension if no
             * patch at Connecting Ridge passes the selection rules. */
        @Rationale { text = "It combines strong published illumination with an image-enhanced terrain model."; }
        @SpecSource { reference = "MVP spec, Demo site selection"; basis = "Split from the site selection statements; fallback added from the spec's prose"; }
    }

    requirement <'REQ-MVP7.2'> patchSelectionRules : SimRequirement {
        doc /* The 2 km patch and scenario date shall be chosen by applying the spec's site selection
             * rules in order. */
        @Rationale { text = "Fixed rules make the choice of patch traceable rather than cherry-picked. This is a requirement on how the scenario is set up."; }
        @SpecSource { reference = "MVP spec, Demo site selection"; basis = "Split from the site selection statements"; }
    }

    requirement <'REQ-MVP8'> r2Framing : PresentationRequirement {
        doc /* Every presentation of scenario R2 shall describe it as selected to illustrate the
             * problem, not as a typical trip. */
        @Rationale { text = "R2 is chosen because a shadow matters, so it is not representative."; }
        @SpecSource { reference = "MVP spec, Demo scenarios"; basis = "Reworded from the Demo scenarios statement"; }
    }

    requirement <'REQ-MVP9'> precomputedRunSet : SimRequirement {
        doc /* The system shall precompute and store a run for every viewer control combination: 3
             * no-road and 9 road runs per scenario, 24 in total. */
        @Rationale { text = "The viewer can then change controls without replanning, in any product form."; }
        @SpecSource { reference = "MVP spec, Demo scenarios (run set)"; basis = "Reworded from the run set statement"; }
        require constraint { sim.display.storedRuns == 24 }
    }

    requirement <'REQ-MVP10'> displayIndependence : SimRequirement {
        doc /* The system shall generate and store the run set without the final display installed. */
        @Rationale { text = "The product form is not yet decided."; }
        @SpecSource { reference = "MVP spec, Handover"; basis = "Reworded from 'shall stay independent of the final display' ('usable' made testable)"; }
    }

    requirement <'REQ-MVP11'> repeatableRuns : SimRequirement {
        doc /* Given the same scenario file and random seed, the system shall produce identical run
             * results. */
        @Rationale { text = "Repeatable runs let reviewers reproduce any result and confirm that only the road changed."; }
        @SpecSource { reference = "MVP spec, MVP checks (CK-1)"; basis = "New"; }
    }

    requirement <'REQ-MVP12'> failureConditions : SimRequirement {
        doc /* The system shall mark a run as failed when charge falls below reserve, the rover
             * enters a cell above the slope limit, or the rover has not arrived by the end of the
             * scenario. */
        @Rationale { text = "Pass or fail is a reported result, so its conditions must be defined."; }
        @SpecSource { reference = "MVP spec, Demo scenarios (failure conditions)"; basis = "New ID for a spec statement"; }
    }

    requirement <'REQ-MVP13.1'> solarReferenceCase : SimRequirement {
        doc /* With the Sun at 0° elevation facing a rear corner, the power model shall give 450 W of
             * solar input, within 5%. */
        @Rationale { text = "Sets a pass tolerance and condition for the spec's 'about 450 W'."; }
        @SpecSource { reference = "MVP spec, MVP checks (CK-3)"; basis = "New"; }
        @StatusInfo { status = StatusKind::tbr; }
    }

    requirement <'REQ-MVP13.2'> drainReferenceCase : SimRequirement {
        doc /* With a constant 600 W load and no solar input, the power model shall take a full
             * battery to empty in 9.0 h, within 5%. */
        @Rationale { text = "Sets a pass tolerance and condition for the spec's 'about 9 h' (5,420 Wh at 600 W)."; }
        @SpecSource { reference = "MVP spec, MVP checks (CK-3)"; basis = "New"; }
        @StatusInfo { status = StatusKind::tbr; }
    }

    requirement <'REQ-MVP14'> shadowCredibility : SimRequirement {
        doc /* At least 80% of the shadow-edge speeds sampled along both runs' routes during the
             * scenario shall fall within the published 0.1 to 1.8 cm/s. */
        @Rationale { text = "Sets a pass threshold for the spec's 'most shadow-edge speeds'."; }
        @SpecSource { reference = "MVP spec, MVP checks (CK-4)"; basis = "New"; }
        @StatusInfo { status = StatusKind::tbr; }
    }


    // ---------------------------------------------------------------------
    // Interface requirements
    // ---------------------------------------------------------------------

    requirement <'REQ-IF01'> terrainInterface : SimRequirement {
        doc /* The environment model shall receive terrain height, slope and wider horizon terrain as
             * GeoTIFF on the LOLA polar stereographic grid, in metres. */
        @Rationale { text = "Defined interfaces let each model be built and tested separately."; }
        @SpecSource { reference = "MVP spec, Interfaces (IF-01)"; basis = "New, from the IF-01 row"; }
    }

    requirement <'REQ-IF02'> ephemerisInterface : SimRequirement {
        doc /* The environment model shall receive time and Sun and Earth azimuth and elevation in
             * degrees, from SPICE kernels or a table. */
        @Rationale { text = "Defined interfaces let each model be built and tested separately. The format is a proposed default."; }
        @SpecSource { reference = "MVP spec, Interfaces (IF-02)"; basis = "New, from the IF-02 row"; }
        @StatusInfo { status = StatusKind::tbr; }
    }

    requirement <'REQ-IF03'> environmentMapsInterface : SimRequirement {
        doc /* The environment model shall provide slope, lit flag, Earth flag and road flag by cell
             * and time, as gridded arrays, to the planner, command link and rover. */
        @Rationale { text = "Defined interfaces let each model be built and tested separately. The format is a proposed default."; }
        @SpecSource { reference = "MVP spec, Interfaces (IF-03)"; basis = "New, from the IF-03 row"; }
        @StatusInfo { status = StatusKind::tbr; }
    }

    requirement <'REQ-IF04'> commandInterface : SimRequirement {
        doc /* The planner shall send each command, with command ID, target x and y, heading and
             * mode, as a JSON message to the rover through the command delay. */
        @Rationale { text = "Routing commands through the delay is what makes communication delay part of the comparison. The format is a proposed default."; }
        @SpecSource { reference = "MVP spec, Interfaces (IF-04)"; basis = "New, from the IF-04 row"; }
        @StatusInfo { status = StatusKind::tbr; }
    }

    requirement <'REQ-IF05'> statusInterface : SimRequirement {
        doc /* The rover shall send each status, with time, x, y, heading, state of charge, mode and
             * command status, as a JSON message to the planner through the command delay. */
        @Rationale { text = "The planner only knows the rover state that reaches it through the delay. The format is a proposed default."; }
        @SpecSource { reference = "MVP spec, Interfaces (IF-05)"; basis = "New, from the IF-05 row"; }
        @StatusInfo { status = StatusKind::tbr; }
    }

    requirement <'REQ-IF06'> runLogInterface : SimRequirement {
        doc /* The planner and rover shall write the route, state, solar power, load, events and
             * results to the display and log, as Parquet or CSV. */
        @Rationale { text = "The stored run set and the display both depend on this log. The format is a proposed default."; }
        @SpecSource { reference = "MVP spec, Interfaces (IF-06)"; basis = "New, from the IF-06 row"; }
        @StatusInfo { status = StatusKind::tbr; }
    }

    requirement <'REQ-IF07'> scenarioInterface : SimRequirement {
        doc /* Every block shall read the road setting, preset, start, goal, start time, seed and
             * parameters from one versioned YAML scenario file. */
        @Rationale { text = "One versioned file keeps every block on the same settings and makes runs repeatable. The format is a proposed default."; }
        @SpecSource { reference = "MVP spec, Interfaces (IF-07)"; basis = "New, from the IF-07 row"; }
        @StatusInfo { status = StatusKind::tbr; }
    }


    // ---------------------------------------------------------------------
    // Model 1: Mobility
    // ---------------------------------------------------------------------

    requirement <'REQ-M1.1'> offRoadMobility : SimRequirement {
        doc /* On off-road cells, the mobility model shall use the no-road speed made good and slip
             * values. */
        @Rationale { text = "Off-road behaviour is the baseline the road is compared against."; }
        @SpecSource { reference = "MVP spec, Simulation models 1"; basis = "Split from REQ-M1"; }
    }

    requirement <'REQ-M1.2'> roadMobility : SimRequirement {
        doc /* On road cells, the mobility model shall use the active road preset's speed made good
             * and slip values. */
        @Rationale { text = "The road's effect enters the simulation only through the preset values."; }
        @SpecSource { reference = "MVP spec, Simulation models 1"; basis = "Split from REQ-M1"; }
    }

    requirement <'REQ-M2'> slopeLimit : SimRequirement {
        doc /* The planner shall reject cells above the 15° slope limit. */
        @Rationale { text = "VIPER's published slope limit (sources B and C)."; }
        @SpecSource { reference = "MVP spec, Simulation models 1"; basis = "Spec"; }
        require constraint { sim.planner.maxPlannedSlope <= 15 ['°'] }
    }

    requirement <'REQ-M3.1'> headingOffsetLimit : SimRequirement {
        doc /* While driving, the rover's body heading shall stay within ±50° of its direction of
             * travel. */
        @Rationale { text = "Allows the rover to turn its panels towards the Sun while driving; the 50° is assumed."; }
        @SpecSource { reference = "MVP spec, Simulation models 1"; basis = "Split from REQ-M3"; }
        require constraint { sim.rover.mobility.maxHeadingOffset <= 50 ['°'] }
    }

    requirement <'REQ-M3.2'> sunlitHeading : SimRequirement {
        doc /* In sunlight, the rover shall choose the allowed heading with the most solar input. */
        @Rationale { text = "Maximises charging while driving."; }
        @SpecSource { reference = "MVP spec, Simulation models 1"; basis = "Split from REQ-M3"; }
    }

    requirement <'REQ-M3.3'> shadowHeading : SimRequirement {
        doc /* In shadow, the rover shall face its direction of travel. */
        @Rationale { text = "There is no solar input to gain by turning."; }
        @SpecSource { reference = "MVP spec, Simulation models 1"; basis = "Split from REQ-M3"; }
    }


    // ---------------------------------------------------------------------
    // Model 2: Power
    // ---------------------------------------------------------------------

    requirement <'REQ-P2.1'> chargeBounds : SimRequirement {
        doc /* The power model shall keep state of charge between 0 and 1. */
        @Rationale { text = "Charge outside this range has no physical meaning."; }
        @SpecSource { reference = "MVP spec, Simulation models 2"; basis = "Split from REQ-P2"; }
        require constraint { sim.rover.power.stateOfCharge >= 0.0 and sim.rover.power.stateOfCharge <= 1.0 }
    }

    requirement <'REQ-P2.2'> excessSolarDiscarded : SimRequirement {
        doc /* The power model shall discard solar power that a full battery cannot accept. */
        @Rationale { text = "Stops a full battery being credited with energy it cannot store."; }
        @SpecSource { reference = "MVP spec, Simulation models 2"; basis = "Split from REQ-P2"; }
    }


    // ---------------------------------------------------------------------
    // Model 3: Environment and road maps
    // ---------------------------------------------------------------------

    requirement <'REQ-E1'> visibilityFromTerrain : SimRequirement {
        doc /* The environment model shall calculate Sun and Earth visibility for each cell from the
             * terrain horizon and the ephemeris. */
        @Rationale { text = "Real terrain shadows are what make the comparison credible."; }
        @SpecSource { reference = "MVP spec, Simulation models 3"; basis = "Reworded from 'calculated from terrain, not hand-painted'"; }
    }

    requirement <'REQ-E3'> roadOnlyChangesFlags : SimRequirement {
        doc /* Turning the road off shall change only the road flags in the environment maps. */
        @Rationale { text = "The road must be the only difference between the runs."; }
        @SpecSource { reference = "MVP spec, Simulation models 3"; basis = "Reworded from REQ-E3"; }
    }


    // ---------------------------------------------------------------------
    // Model 4: Command delay
    // ---------------------------------------------------------------------

    requirement <'REQ-C1.1'> delaySampling : SimRequirement {
        doc /* The command link shall sample each round-trip delay uniformly from 6 to 10 s. */
        @Rationale { text = "The published round-trip range (source B). A uniform distribution is assumed."; }
        @SpecSource { reference = "MVP spec, Simulation models 4"; basis = "Split from REQ-C1"; }
        @StatusInfo { status = StatusKind::tbr; }
        require constraint { sim.commLink.minRoundTrip == 6 [s] and sim.commLink.maxRoundTrip == 10 [s] }
    }

    requirement <'REQ-C1.2'> statusPeriod : SimRequirement {
        doc /* The rover shall send a status message every 60 s. */
        @Rationale { text = "Sets how quickly the planner learns the rover state; the 60 s is assumed."; }
        @SpecSource { reference = "MVP spec, Simulation models 4"; basis = "Split from REQ-C1"; }
        require constraint { sim.commLink.statusPeriod == 60 [s] }
    }

    requirement <'REQ-C2.1'> commandBeforeWaypoint : SimRequirement {
        doc /* The rover shall not start a new waypoint without a received command. */
        @Rationale { text = "The rover drives only on commands from the planner."; }
        @SpecSource { reference = "MVP spec, Simulation models 4"; basis = "Split from REQ-C2"; }
    }

    requirement <'REQ-C2.2'> earthBeforeWaypoint : SimRequirement {
        doc /* The rover shall not start a new waypoint while Earth is out of view. */
        @Rationale { text = "Commands need a direct-to-Earth link."; }
        @SpecSource { reference = "MVP spec, Simulation models 4"; basis = "Split from REQ-C2"; }
    }

    requirement <'REQ-C2.3'> commandResend : SimRequirement {
        doc /* The planner shall resend a command for which no acknowledgement has arrived within 15
             * s. */
        @Rationale { text = "A dropped command must not stall the trip. The timeout must exceed the 10 s maximum round trip, or every command would be resent; the spec's 10 s is raised to 15 s."; }
        @SpecSource { reference = "MVP spec, Simulation models 4"; basis = "Split from REQ-C2; timeout raised from 10 s"; }
        @StatusInfo { status = StatusKind::tbr; }
        require constraint { sim.commLink.ackTimeout == 15 [s] }
    }

    requirement <'REQ-C2.4'> statusReplacement : SimRequirement {
        doc /* The rover shall not resend a dropped status message. */
        @Rationale { text = "Status is periodic, so the next scheduled message supersedes a lost one."; }
        @SpecSource { reference = "MVP spec, Simulation models 4"; basis = "Split from REQ-C2"; }
    }

    requirement <'REQ-C2.5'> dropAndResendLogging : SimRequirement {
        doc /* The system shall log every dropped and resent message. */
        @Rationale { text = "Drops and resends feed the reliability result (REQ-MVP1.4)."; }
        @SpecSource { reference = "MVP spec, Simulation models 4"; basis = "Split from REQ-C2"; }
    }

    requirement <'REQ-C2.6'> commandAcknowledgement : SimRequirement {
        doc /* The rover shall send an acknowledgement through the command delay as soon as it
             * receives a command. */
        @Rationale { text = "REQ-C2.3 needs an acknowledgement; the 60 s status message is too slow to serve as one."; }
        @SpecSource { reference = "MVP spec, Simulation models 4"; basis = "New"; }
        @StatusInfo { status = StatusKind::tbr; }
    }


    // ---------------------------------------------------------------------
    // Model 5: Planner
    // ---------------------------------------------------------------------

    requirement <'REQ-R1'> timeAndEnergy : SimRequirement {
        doc /* For each route it evaluates, the planner shall calculate travel time and energy from
             * the active speed, slip and load. */
        @Rationale { text = "Time and energy are what the road changes."; }
        @SpecSource { reference = "MVP spec, Simulation models 5"; basis = "Reworded from REQ-R1"; }
    }

    requirement <'REQ-R2.1'> reserveLimit : SimRequirement {
        doc /* The planner shall not plan driving that takes state of charge below the battery
             * reserve. */
        @Rationale { text = "Falling below reserve fails the run (REQ-MVP12)."; }
        @SpecSource { reference = "MVP spec, Simulation models 5"; basis = "Split from REQ-R2; the slope limit part is REQ-M2"; }
    }

    requirement <'REQ-R2.2'> earthLimit : SimRequirement {
        doc /* The planner shall not plan driving while Earth is out of view. */
        @Rationale { text = "Driving needs a direct-to-Earth link."; }
        @SpecSource { reference = "MVP spec, Simulation models 5"; basis = "Split from REQ-R2"; }
    }

    requirement <'REQ-R3'> routeCost : SimRequirement {
        doc /* Among the routes it evaluates, the planner shall choose the one with the lowest cost,
             * where cost is arrival time plus 0.001 h per Wh drawn from the battery. */
        @Rationale { text = "A stated cost function can be checked; 'minimise' alone cannot. The energy penalty is assumed."; }
        @SpecSource { reference = "MVP spec, Simulation models 5"; basis = "Reworded from REQ-R3 ('minimise arrival time')"; }
    }

    requirement <'REQ-R4.1'> planTime : SimRequirement {
        doc /* The planner shall finish each plan within 5 min of wall-clock time. */
        @Rationale { text = "Matches the 5 min command cycle; the value is assumed."; }
        @SpecSource { reference = "MVP spec, Simulation models 5"; basis = "Split from REQ-R4"; }
        require constraint { sim.planner.maxPlanTime <= 5 [min] }
    }

    requirement <'REQ-R4.2'> fullResolutionRecheck : SimRequirement {
        doc /* The planner shall recheck the final route at full terrain resolution. */
        @Rationale { text = "A coarse-grid plan could miss a steep or shadowed cell."; }
        @SpecSource { reference = "MVP spec, Simulation models 5"; basis = "Split from REQ-R4"; }
    }


    // Not modelled: the handover duty that the simulation engineer shall propose tools,
    // schedule, budget and code ownership. It is a project duty, not a system requirement.

    // ---------------------------------------------------------------------
    // Specification groups, used for the satisfy relations
    // ---------------------------------------------------------------------

    requirement simulationSpecification : SimRequirement {
        doc /* All simulation requirements, under the modelling assumptions below. */
        assume constraint <'ASM-P1'> loadModel {
            doc /* Load is the awake load plus driving power multiplied by the moving fraction. (Was
                 * REQ-P1; it describes how the model works, not a need.) */
        }
        assume constraint <'ASM-E2'> visibilityHeights {
            doc /* Earth visibility is evaluated 2 m above the surface and sunlight 1 m above it. (Was
                 * REQ-E2; modelling choice, values assumed.) */
            sim.environment.earthVisibilityHeight == 2 [m] and sim.environment.sunlightHeight == 1 [m]
        }
        assume constraint <'ASM-C1'> timedMessageEvents {
            doc /* Message delays are handled as timed events. (Was part of REQ-C1; implementation
                 * choice.) */
        }
        require roadComparison;
        require reportedResults;
        require riskAndReliability;
        require reliabilityMeasure;
        require comparisonDifference;
        require assumedValuesConfigurable;
        require fixedScenario;
        require widerTerrainForShadows;
        require demoSite;
        require patchSelectionRules;
        require precomputedRunSet;
        require displayIndependence;
        require repeatableRuns;
        require failureConditions;
        require solarReferenceCase;
        require drainReferenceCase;
        require shadowCredibility;
        require terrainInterface;
        require ephemerisInterface;
        require environmentMapsInterface;
        require commandInterface;
        require statusInterface;
        require runLogInterface;
        require scenarioInterface;
        require offRoadMobility;
        require roadMobility;
        require slopeLimit;
        require headingOffsetLimit;
        require sunlitHeading;
        require shadowHeading;
        require chargeBounds;
        require excessSolarDiscarded;
        require visibilityFromTerrain;
        require roadOnlyChangesFlags;
        require delaySampling;
        require statusPeriod;
        require commandBeforeWaypoint;
        require earthBeforeWaypoint;
        require commandResend;
        require statusReplacement;
        require dropAndResendLogging;
        require commandAcknowledgement;
        require timeAndEnergy;
        require reserveLimit;
        require earthLimit;
        require routeCost;
        require planTime;
        require fullResolutionRecheck;
    }

    requirement presentationSpecification : PresentationRequirement {
        doc /* Requirements on how simulation results are presented. */
        require reportEitherWay;
        require caveatSimplifiedModel;
        require caveatAutonomy;
        require r2Framing;
    }
}


## 04_Behaviour

In [ ]:
package LRD_Behaviour {
    doc /*
     * Functional flow of one simulation run, and the rover's operating modes.
     * The failure conditions come from the spec: charge below reserve, a cell above
     * the slope limit, or no arrival by the end of the scenario.
     */

    public import LRD_Structure::*;

    // =====================================================================
    // One command cycle of a run: scenario in, run record out.
    // A full run repeats the plan, delay and drive steps every 5 min until the
    // rover arrives or fails; status returns to the planner through the delay (IF-05).
    // Not modelled as behaviour: command acknowledgement and resend (REQ-C2.3, C2.6,
    // verified by CK-6) and the minimum 5 min between commands.
    // =====================================================================

    action def RunScenario {
        in item settings : ScenarioSettings;
        out item record : RunRecord;

        action loadScenario {
            in item settings : ScenarioSettings = RunScenario::settings;
            out item config : ScenarioSettings;
        }

        flow from loadScenario.config to computeMaps.config;

        then action computeMaps {
            doc /* Sun and Earth visibility per cell from terrain horizons; road flags from the centreline. */
            in item config : ScenarioSettings;
            out item maps : EnvironmentMaps;
        }

        flow from computeMaps.maps to planRoute.maps;

        then action planRoute {
            doc /* Time-aware plan minimising arrival time plus an energy penalty, within hard limits. */
            in item maps : EnvironmentMaps;
            out item command : Command;
        }

        flow from planRoute.command to delayCommand.commandIn;

        then action delayCommand {
            doc /* Hold the command for half of a sampled 6 to 10 s round trip; optionally drop it. */
            in item commandIn : Command;
            out item commandOut : Command;
        }

        flow from delayCommand.commandOut to driveAndUpdatePower.command;

        then action driveAndUpdatePower {
            doc /* Move with slip, update battery as solar input minus load, and report status. */
            in item command : Command;
            out item status : RoverStatus;
        }

        flow from driveAndUpdatePower.status to logResults.status;

        then action logResults {
            doc /* Record route, state, solar power, load, events and the reported results. */
            in item status : RoverStatus;
            out item record : RunRecord = RunScenario::record;
        }
    }

    // A comparison is the same scenario run twice, changing only the road setting.
    action def RunComparison {
        in item settings : ScenarioSettings;
        action noRoadRun : RunScenario;
        then action roadRun : RunScenario;
    }

    // =====================================================================
    // Rover operating modes
    // =====================================================================

    attribute def CommandReceived;
    attribute def HoldCommanded;
    attribute def WaypointReached;
    attribute def GoalReached;
    attribute def EarthLost;
    attribute def EarthRegained;
    attribute def ChargeBelowReserve;
    attribute def SlopeLimitExceeded;
    attribute def ScenarioTimeout;

    state def RoverModes {
        first start then awaitingCommand;

        state awaitingCommand {
            doc /* No new waypoint starts without a command (REQ-C2.1). */
        }
        accept CommandReceived then driving;
        accept HoldCommanded then holding;
        accept EarthLost then waitingForEarth;
        accept ChargeBelowReserve then failed;
        accept ScenarioTimeout then failed;

        state driving;
        accept WaypointReached then awaitingCommand;
        accept GoalReached then arrived;
        accept ChargeBelowReserve then failed;
        accept SlopeLimitExceeded then failed;
        accept EarthLost then waitingForEarth;
        accept ScenarioTimeout then failed;

        state holding {
            doc /* Planner chose to wait, for example for a moving shadow to pass. */
        }
        accept CommandReceived then driving;
        accept EarthLost then waitingForEarth;
        accept ChargeBelowReserve then failed;
        accept ScenarioTimeout then failed;

        state waitingForEarth {
            doc /* No driving without Earth in view (REQ-R2.2, C2.2). Entered from awaiting a command,
                 * holding or mid-drive; the rover stops until Earth is back in view. */
        }
        accept EarthRegained then awaitingCommand;
        accept ChargeBelowReserve then failed;
        accept ScenarioTimeout then failed;

        state arrived;
        state failed;
    }
}


## 05_RoadPresets_Variants

In [ ]:
package LRD_Variants {
    doc /*
     * Variability: the road presets and awake-load presets the viewer can choose.
     * All road values are assumed placeholders until test data or a source exists.
     * Trip times assume a straight 1.5 km route with no detours or waiting.
     * The stored run set is 3 no-road plus 9 road runs per scenario, 24 in total.
     */

    private import ScalarValues::*;
    private import ISQ::*;
    private import SI::*;
    public import LRD_Structure::*;

    part def RoadPreset {
        attribute speedMadeGoodFlat : SpeedValue;
        attribute slipMultiplier : Real;
        attribute rollingResistance : Real;
        attribute maxWaypointSpacing : LengthValue;
        attribute tripTimeStraight1500m : DurationValue;
    }

    // No-road speed made good and waypoint spacing are sourced ('='). Every other
    // road value is an assumed placeholder ('default'), and trip times are derived
    // from the assumed speeds.

    part noRoad : RoadPreset {
        attribute :>> speedMadeGoodFlat = 0.8 [cm/s];
        attribute :>> slipMultiplier default 1.0;
        attribute :>> rollingResistance default 0.2;
        attribute :>> maxWaypointSpacing = 8 [m];
        attribute :>> tripTimeStraight1500m default 52 [h];   // derived
    }

    part halfBenefit : RoadPreset {
        attribute :>> speedMadeGoodFlat default 1.2 [cm/s];
        attribute :>> slipMultiplier default 0.75;
        attribute :>> rollingResistance default 0.15;
        attribute :>> maxWaypointSpacing default 8 [m];
        attribute :>> tripTimeStraight1500m default 35 [h];   // derived
    }

    part expectedBenefit : RoadPreset {
        attribute :>> speedMadeGoodFlat default 1.6 [cm/s];
        attribute :>> slipMultiplier default 0.5;
        attribute :>> rollingResistance default 0.1;
        attribute :>> maxWaypointSpacing default 8 [m];
        attribute :>> tripTimeStraight1500m default 26 [h];   // derived
    }

    part highBenefit : RoadPreset {
        attribute :>> speedMadeGoodFlat default 3.2 [cm/s];
        attribute :>> slipMultiplier default 0.25;
        attribute :>> rollingResistance default 0.05;
        attribute :>> maxWaypointSpacing default 16 [m];
        attribute :>> tripTimeStraight1500m default 13 [h];   // derived
    }

    variation part def RoadPresetChoices :> RoadPreset {
        variant noRoad;
        variant halfBenefit;
        variant expectedBenefit;
        variant highBenefit;
    }

    variation attribute def AwakeLoadChoices :> PowerValue {
        variant attribute lowLoad = 150 [W];
        variant attribute nominalLoad = 250 [W];
        variant attribute highLoad = 350 [W];
    }

    enum def ScenarioKind {
        doc /* R1 baseline with no moving shadow on the main route; R2 with a real moving shadow. */
        R1_Baseline;
        R2_MovingShadow;
    }

    // The family of runs the viewer can select from
    abstract part demoRunFamily : LunarRoadDemoSystem {
        part roadPreset : RoadPresetChoices[1];
        attribute awakeLoadPreset : AwakeLoadChoices[1];
        attribute scenarioKind : ScenarioKind;
    }

    // Two example configurations from the run set
    part r1NoRoadNominal :> demoRunFamily {
        part redefines roadPreset = roadPreset::noRoad;
        attribute redefines awakeLoadPreset = awakeLoadPreset::nominalLoad;
        attribute redefines scenarioKind = ScenarioKind::R1_Baseline;
    }

    part r2ExpectedRoadNominal :> demoRunFamily {
        part redefines roadPreset = roadPreset::expectedBenefit;
        attribute redefines awakeLoadPreset = awakeLoadPreset::nominalLoad;
        attribute redefines scenarioKind = ScenarioKind::R2_MovingShadow;
    }
}


## 06_Hardware_1to10

In [ ]:
package LRD_Hardware {
    doc /*
     * Physical architecture of the 1:10 VIPER-look tabletop rover, and its mapping
     * to VIPER's published electrical blocks.
     * Sources: "Lunar Road Demo: 1:10 VIPER-Look Rover Hardware Design" and
     * "VIPER Electrical Architecture Reconstruction".
     * The model borrows VIPER block names only; it reproduces none of VIPER's design.
     */

    private import ScalarValues::*;
    private import ISQ::*;
    private import SI::*;
    public import LRD_Structure::*;

    // =====================================================================
    // Ports for the tabletop electrical design (connections E1 to E6)
    // =====================================================================

    port def PowerRail {
        doc /* 4.8 V battery bus, up to about 5.8 V off the charger. */
        out attribute voltage : ElectricPotentialDifferenceValue;
    }
    port def PwmPair      { doc /* 2 PWM lines per motor, plus shared enable (SLP). */ }
    port def MotorPower   { doc /* Motor power, from a driver channel to its gearmotor. */ }
    port def EncoderLink  { doc /* Encoder A and B back to the Pico (GP10 to GP17), powered from its 3V3.
                                 * Shares the wheel's 6-pin JST-SH cable with motor power. */ }
    port def BatterySense { doc /* Battery voltage through a 2:1 divider to GP26 (ADC0). */ }
    port def I2CBus       { doc /* I2C plus 3V3. */ }
    port def LedDrive     { doc /* One GPIO to a transistor; LEDs run from the battery rail. */ }
    item def TabletopStatus {
        doc /* The MVP's IF-05 fields, with battery voltage replacing state of charge. */
        attribute time : DurationValue;
        attribute x : LengthValue;
        attribute y : LengthValue;
        attribute heading : AngularMeasureValue;
        attribute mode : String;
        attribute commandStatus : String;
        attribute batteryVoltage : ElectricPotentialDifferenceValue;
    }

    port def WifiLink {
        doc /* Wi-Fi to a laptop; commands reuse the MVP's IF-04 fields, status uses TabletopStatus. */
        in item command : Command;
        out item status : TabletopStatus;
    }

    // =====================================================================
    // Parts
    // =====================================================================

    part def BatteryBox {
        doc /* 4 AA 2,500 mAh NiMH cells in a switched box. */
        attribute nominalVoltage : ElectricPotentialDifferenceValue = 4.8 [V];
        attribute energy : EnergyValue = 12 [W*h];
        port powerOut : PowerRail;
        port sense : BatterySense;
    }

    part def PicoController {
        doc /* Raspberry Pi Pico 2 W: command, data and control on one board. */
        attribute speedLoopRate : Real default 50.0;   // Hz, assumed
        attribute commsLossTimeout : DurationValue = 1 [s];
        port powerIn : ~PowerRail;
        port pwmDriver1 : PwmPair;
        port pwmDriver2 : PwmPair;
        port imuBus : I2CBus;
        port ledOut : LedDrive;
        port wifi : WifiLink;
        port batterySense : ~BatterySense;
        port encoderFL : ~EncoderLink;
        port encoderFR : ~EncoderLink;
        port encoderRL : ~EncoderLink;
        port encoderRR : ~EncoderLink;
    }

    part def MotorDriver {
        doc /* Adafruit DRV8833 breakout, 1.2 A continuous per channel, 2 motors each. */
        attribute continuousCurrentPerChannel : ElectricCurrentValue = 1.2 [A];
        port powerIn : ~PowerRail;
        port pwmIn : ~PwmPair;
        port motorA : MotorPower;
        port motorB : MotorPower;
    }

    part def WheelModule {
        doc /* 1006:1 micro metal gearmotor with Hall encoder and a printed grousered wheel. */
        attribute wheelDiameter : LengthValue = 50 [mm];
        attribute wheelWidth : LengthValue = 20 [mm];
        attribute stallTorqueDerated : TorqueValue = 0.23 [N*m];   // derived from 2.3 kg.cm at 4.8 V
        port motorIn : ~MotorPower;
        port encoder : EncoderLink;
    }

    part def Imu {
        doc /* 6-axis IMU breakout; heading by gyro only, so it drifts. */
        port bus : ~I2CBus;
    }

    part def MastLeds {
        doc /* Two white LEDs in the mast; camera holes are dummies. */
        port drive : ~LedDrive;
    }

    part def ChassisTub    { doc /* Load-carrying printed tub, 150 x 118 x 30 mm, 17 mm ground clearance. */ }
    part def BodyShell     { doc /* Cosmetic open-bottom shell with top hatch. */ }
    part def DummyPanel    { doc /* Printed solar panel with a cell grid; no power generation. */ }
    part def Mast          { doc /* T-shaped mast, top at 250 mm. */ }

    part def TabletopRover {
        doc /* 1:10 four-wheel skid-steer model, about 0.6 kg, top speed near 3 cm/s. */

        // Measured or calculated properties checked by the CK-H checks
        attribute footprint : LengthValue = 170 [mm];   // design value
        attribute mass : MassValue default 0.6 [kg];
        attribute topSpeed : SpeedValue = 3.0 [cm/s];
        attribute scaleSpeed : SpeedValue = 2 [cm/s];
        attribute drivingPower : PowerValue default 1.48 [W];   // derived from assumed Pico and LED/IMU loads
        attribute partsCostGBP : Real default 173.70;
        attribute measuredFootprint : LengthValue;
        attribute measuredWheelDiameter : LengthValue;
        attribute measuredWheelWidth : LengthValue;
        attribute measuredLevelSpeed : SpeedValue;
        attribute continuousRunTime : DurationValue;
        attribute commsLossStopTime : DurationValue;
        attribute maxClimbableSlope : AngularMeasureValue;

        // Structure
        part tub : ChassisTub;
        part shell : BodyShell;
        part panels : DummyPanel[3];
        part mast : Mast;

        // Electrical
        part battery : BatteryBox;
        part controller : PicoController;
        part driver1 : MotorDriver;
        part driver2 : MotorDriver;
        part frontLeft : WheelModule;
        part frontRight : WheelModule;
        part rearLeft : WheelModule;
        part rearRight : WheelModule;
        part imu : Imu;
        part leds : MastLeds;

        port wifi : WifiLink;
        bind wifi = controller.wifi;

        // E1: battery to both drivers directly, to the Pico's VSYS through a silicon diode,
        // and to the battery sense pin through a 2:1 divider
        connection 'E1 battery to driver 1' connect battery.powerOut to driver1.powerIn;
        connection 'E1 battery to driver 2' connect battery.powerOut to driver2.powerIn;
        connection 'E1 battery to Pico VSYS' connect battery.powerOut to controller.powerIn;
        connection 'E1 battery sense' connect battery.sense to controller.batterySense;

        // E2: Pico to drivers, 8 PWM lines and 1 shared enable
        connection 'E2 Pico to driver 1' connect controller.pwmDriver1 to driver1.pwmIn;
        connection 'E2 Pico to driver 2' connect controller.pwmDriver2 to driver2.pwmIn;

        // E3: control board to each wheel over one 6-pin JST-SH cable, carrying motor
        // power from a driver and encoder signals back to the Pico
        connection 'E3 front left power'  connect driver1.motorA to frontLeft.motorIn;
        connection 'E3 front right power' connect driver1.motorB to frontRight.motorIn;
        connection 'E3 rear left power'   connect driver2.motorA to rearLeft.motorIn;
        connection 'E3 rear right power'  connect driver2.motorB to rearRight.motorIn;
        connection 'E3 front left encoder'  connect frontLeft.encoder to controller.encoderFL;
        connection 'E3 front right encoder' connect frontRight.encoder to controller.encoderFR;
        connection 'E3 rear left encoder'   connect rearLeft.encoder to controller.encoderRL;
        connection 'E3 rear right encoder'  connect rearRight.encoder to controller.encoderRR;

        // E4 and E5
        connection 'E4 Pico to IMU'  connect controller.imuBus to imu.bus;
        connection 'E5 Pico to LEDs' connect controller.ledOut to leds.drive;
    }

    part def RoutePlayerLaptop {
        doc /* Reads a stored route (IF-06 format) and sends waypoints; can hold each message for
             * half of a sampled 6 to 10 s round trip. */
        port wifi : ~WifiLink;
    }

    part def OverheadCamera {
        doc /* USB webcam and printed hatch marker for position and heading correction (decision D3). */
    }

    part tabletopDemo {
        part rover : TabletopRover;
        part laptop : RoutePlayerLaptop;
        part camera : OverheadCamera;
        // E6: Pico to laptop over Wi-Fi
        connection 'E6 Wi-Fi link' connect rover.wifi to laptop.wifi;
    }

    // =====================================================================
    // VIPER reference blocks (public sources only; not simulated, not reproduced)
    // =====================================================================

    package ViperReference {
        part def SolarArray       { attribute powerPerPanel : PowerValue = 320 [W]; }
        part def ViperBattery     { doc /* One of two internal units; core components shared with Orion. */ }
        part def IntegratedAvionicsUnit { doc /* Command and data handling with electric power functions (Moog). */ }
        part def SEPIA            { doc /* With the IAU, manages solar array, battery charging and power distribution. */ }
        part def MotorControlBox  { doc /* Controls 6 BLDC actuators; 20 A peak per axis for 10 s; in the warm box. */ }
        part def ViperWheelModule { doc /* Drive, steering and suspension actuators. */ }
        part def ViperImu;
        part def StarTracker      { doc /* Rocket Lab ST16RT2; 28 V dual RS-485. */ }
        part def CameraSystem     { doc /* Eight cameras with a Camera Control Unit. */ }
        part def LightSystem      { doc /* Eight LED-array lights with a Light Controller. */ }
        part def XBandLink        { doc /* X-band direct to Earth, gimballed high-gain and fixed low-gain antennas. */ }

        part def ViperRover {
            attribute batteryCapacityStartOfLife : EnergyValue = 5420 [W*h];   // total of both units, 0 C
            part solarArrays : SolarArray[3];
            part batteries : ViperBattery[2];
            part iau : IntegratedAvionicsUnit;
            part sepia : SEPIA;
            part motorControlBoxes : MotorControlBox[1..*];   // count not published; two is an inference
            part wheelModules : ViperWheelModule[4];
            part imu : ViperImu;
            part starTracker : StarTracker;
            part cameras : CameraSystem;
            part lights : LightSystem;
            part radio : XBandLink;
        }
    }

    // =====================================================================
    // Mapping: which tabletop element stands in for which VIPER block
    // =====================================================================

    part viperReference : ViperReference::ViperRover;
    part tabletopRoverDesign : TabletopRover;

    allocation def StandsInFor {
        doc /* The tabletop element plays the role of the VIPER block in the demo only. */
        end viperBlock : Parts::Part;
        end modelElement : Parts::Part;
    }

    // Mapping follows the hardware spec's VIPER-to-tabletop table.
    allocation batteryMap : StandsInFor allocate viperReference.batteries to tabletopRoverDesign.battery;
    allocation avionicsMap : StandsInFor allocate viperReference.iau to tabletopRoverDesign.controller;
    allocation powerMgmtMap : StandsInFor allocate viperReference.sepia to tabletopRoverDesign.controller;
    allocation powerSwitchMap : StandsInFor allocate viperReference.sepia to tabletopRoverDesign.battery;
    // The motor control boxes, whatever their number, stand in as a set for both drivers.
    allocation motorCtrlMap1 : StandsInFor allocate viperReference.motorControlBoxes to tabletopRoverDesign.driver1;
    allocation motorCtrlMap2 : StandsInFor allocate viperReference.motorControlBoxes to tabletopRoverDesign.driver2;
    // VIPER's four wheel modules stand in as a set for the four tabletop wheels.
    allocation wheelMapFL : StandsInFor allocate viperReference.wheelModules to tabletopRoverDesign.frontLeft;
    allocation wheelMapFR : StandsInFor allocate viperReference.wheelModules to tabletopRoverDesign.frontRight;
    allocation wheelMapRL : StandsInFor allocate viperReference.wheelModules to tabletopRoverDesign.rearLeft;
    allocation wheelMapRR : StandsInFor allocate viperReference.wheelModules to tabletopRoverDesign.rearRight;
    allocation imuMap : StandsInFor allocate viperReference.imu to tabletopRoverDesign.imu;
    allocation starTrackerMap : StandsInFor allocate viperReference.starTracker to tabletopRoverDesign.imu;
    allocation camerasMap : StandsInFor allocate viperReference.cameras to tabletopRoverDesign.mast;
    allocation lightsMap : StandsInFor allocate viperReference.lights to tabletopRoverDesign.leds;
    allocation radioMap : StandsInFor allocate viperReference.radio to tabletopRoverDesign.controller;
    allocation panelsMap : StandsInFor allocate viperReference.solarArrays to tabletopRoverDesign.panels;

    use case def 'Replay Route On Tabletop' {
        subject rover : TabletopRover;
        actor laptop : RoutePlayerLaptop;
        objective {
            doc /* Drive a stored road or no-road route on a printed terrain board at 1:1,000 route scale,
                 * as a physical prop for the simulation results. */
        }
    }
}


## 07_Requirements_Hardware

In [ ]:
package LRD_Requirements_Hardware {
    doc /*
     * Requirements for the 1:10 tabletop rover, rewritten to INCOSE rules. REQ-H1 to H10 keep
     * the spec's IDs, with decimal suffixes where a requirement was split. REQ-H11 (the spec's
     * required caveat) and REQ-H12 (charging safety, proposed) were assigned in this model.
     */

    private import ScalarValues::*;
    private import ISQ::*;
    private import SI::*;
    private import ModelingMetadata::*;
    private import LRD_Metadata::*;
    public import LRD_Hardware::*;
    private import LRD_Requirements_Sim::PresentationRequirement;

    requirement def HardwareRequirement {
        subject rover : TabletopRover;
    }

    requirement <'REQ-H1.1'> footprint : HardwareRequirement {
        doc /* The rover shall fit a 170 mm × 170 mm footprint, within ±2 mm. */
        @Rationale { text = "Matches VIPER's footprint at 1:10."; }
        @SpecSource { reference = "Hardware spec, Requirements"; basis = "Split from REQ-H1"; }
        require constraint { rover.measuredFootprint >= 168 [mm] and rover.measuredFootprint <= 172 [mm] }
    }

    requirement <'REQ-H1.2'> wheelSize : HardwareRequirement {
        doc /* Each wheel shall be 50 mm in diameter and 20 mm wide, within ±2 mm. */
        @Rationale { text = "Matches VIPER's wheel size at 1:10."; }
        @SpecSource { reference = "Hardware spec, Requirements"; basis = "Split from REQ-H1"; }
        require constraint { rover.measuredWheelDiameter >= 48 [mm] and rover.measuredWheelDiameter <= 52 [mm] and rover.measuredWheelWidth >= 18 [mm] and rover.measuredWheelWidth <= 22 [mm] }
    }

    requirement <'REQ-H2.1'> boxBody : HardwareRequirement {
        doc /* The rover shall have a box-shaped body. */
        @Rationale { text = "One of VIPER's defining visual features."; }
        @SpecSource { reference = "Hardware spec, Requirements"; basis = "Split from REQ-H2 ('recognisable features' replaced by the list)"; }
    }

    requirement <'REQ-H2.2'> cornerWheels : HardwareRequirement {
        doc /* The rover shall have a wheel at each corner. */
        @Rationale { text = "One of VIPER's defining visual features."; }
        @SpecSource { reference = "Hardware spec, Requirements"; basis = "Split from REQ-H2"; }
    }

    requirement <'REQ-H2.3'> sidePanels : HardwareRequirement {
        doc /* The rover shall have panels on its rear, left and right sides. */
        @Rationale { text = "Shows where VIPER's solar arrays sit."; }
        @SpecSource { reference = "Hardware spec, Requirements"; basis = "Split from REQ-H2"; }
    }

    requirement <'REQ-H2.4'> frontMast : HardwareRequirement {
        doc /* The rover shall have a mast at the front. */
        @Rationale { text = "One of VIPER's defining visual features."; }
        @SpecSource { reference = "Hardware spec, Requirements"; basis = "Split from REQ-H2"; }
    }

    requirement <'REQ-H3'> printable : HardwareRequirement {
        doc /* All structural parts shall print in PLA on a 180 mm × 180 mm bed without supports. */
        @Rationale { text = "Keeps the build possible on a small, common printer."; }
        @SpecSource { reference = "Hardware spec, Requirements"; basis = "Reworded from REQ-H3 (escape clause removed; the CAD section names no exception)"; }
    }

    requirement <'REQ-H4'> levelSpeed : HardwareRequirement {
        doc /* The rover shall hold a commanded speed of 2 cm/s on level ground, within ±10%. */
        @Rationale { text = "VIPER's 20 cm/s top speed at 1:10."; }
        @SpecSource { reference = "Hardware spec, Requirements"; basis = "Spec"; }
        require constraint { rover.measuredLevelSpeed >= 1.8 [cm/s] and rover.measuredLevelSpeed <= 2.2 [cm/s] }
    }

    requirement <'REQ-H5.1'> slopeClimb : HardwareRequirement {
        doc /* The rover shall climb a 15° slope of the demo surface (the printed terrain map on a
             * rigid board, decision D2). */
        @Rationale { text = "Matches the simulation's 15° slope limit."; }
        @SpecSource { reference = "Hardware spec, Requirements; decision D2"; basis = "Split from REQ-H5; 'chosen demo surface' named from D2"; }
        require constraint { rover.maxClimbableSlope >= 15 ['°'] }
    }

    requirement <'REQ-H5.2'> turnOnSpot : HardwareRequirement {
        doc /* The rover shall turn 360° on the spot on the demo surface. */
        @Rationale { text = "Skid steering must work on the board to follow replayed routes."; }
        @SpecSource { reference = "Hardware spec, Requirements; decision D2"; basis = "Split from REQ-H5"; }
    }

    requirement <'REQ-H6'> endurance : HardwareRequirement {
        doc /* Starting from freshly charged cells, the rover shall drive continuously for at least 2
             * hours. */
        @Rationale { text = "Covers a full demo session without changing cells."; }
        @SpecSource { reference = "Hardware spec, Requirements"; basis = "Reworded from REQ-H6 (start condition added from CK-H5)"; }
        require constraint { rover.continuousRunTime >= 2 [h] }
    }

    requirement <'REQ-H7.1'> wifiCommands : HardwareRequirement {
        doc /* The rover shall accept waypoint commands over Wi-Fi. */
        @Rationale { text = "Lets the laptop replay a stored route."; }
        @SpecSource { reference = "Hardware spec, Requirements"; basis = "Split from REQ-H7"; }
    }

    requirement <'REQ-H7.2'> wifiStatus : HardwareRequirement {
        doc /* The rover shall report its position, heading and battery voltage over Wi-Fi. */
        @Rationale { text = "The route player needs feedback to keep the rover on the route."; }
        @SpecSource { reference = "Hardware spec, Requirements"; basis = "Split from REQ-H7"; }
    }

    requirement <'REQ-H8'> safeStop : HardwareRequirement {
        doc /* The rover shall stop within 1 s after 1 s passes with no message received. */
        @Rationale { text = "Stops a runaway rover if the link drops. The route player must therefore send a message at least once a second while a route runs."; }
        @SpecSource { reference = "Hardware spec, Requirements"; basis = "Reworded from REQ-H8 (timeout made explicit, matching the safe-stop design)"; }
        require constraint { rover.commsLossStopTime <= 1 [s] }
    }

    requirement <'REQ-H9'> cost : HardwareRequirement {
        doc /* Total parts cost shall stay under £300, including filament and delivery. */
        @Rationale { text = "Project budget."; }
        @SpecSource { reference = "Hardware spec, Requirements"; basis = "Spec"; }
        require constraint { rover.partsCostGBP < 300.0 }
    }

    requirement <'REQ-H10'> massLimit : HardwareRequirement {
        doc /* The rover's mass shall stay under 0.8 kg. */
        @Rationale { text = "The spec's motor torque margins assume this mass."; }
        @SpecSource { reference = "Hardware spec, Requirements"; basis = "Reworded from REQ-H10 (reason moved to rationale)"; }
        require constraint { rover.mass < 0.8 [kg] }
    }

    requirement <'REQ-H11.1'> scaleModelCaveat : PresentationRequirement {
        doc /* Every display, sign or video shown with the rover shall state that it is a scale model
             * driving a replayed route. */
        @Rationale { text = "Viewers must not mistake the prop for a working rover."; }
        @SpecSource { reference = "Hardware spec, Required caveat"; basis = "Split from the required caveat"; }
    }

    requirement <'REQ-H11.2'> noPerformanceClaims : PresentationRequirement {
        doc /* Every display, sign or video shown with the rover shall state that its speed, slip and
             * energy use say nothing about VIPER or a lunar road. */
        @Rationale { text = "The tabletop rover's physics do not scale to the Moon."; }
        @SpecSource { reference = "Hardware spec, Required caveat"; basis = "Split from the required caveat"; }
    }

    requirement <'REQ-H12'> chargingSafety : HardwareRequirement {
        doc /* The rover shall provide no path for charging the cells while they are fitted. */
        @Rationale { text = "The rover has no charge control, so charging NiMH cells inside it would be a heat risk. Cells are charged in an external NiMH charger."; }
        @SpecSource { reference = "Hardware spec, Parts list (charger)"; basis = "New"; }
        @StatusInfo { status = StatusKind::tbr; }
    }

    requirement hardwareSpecification : HardwareRequirement {
        doc /* All 1:10 rover build requirements. */
        require footprint;
        require wheelSize;
        require boxBody;
        require cornerWheels;
        require sidePanels;
        require frontMast;
        require printable;
        require levelSpeed;
        require slopeClimb;
        require turnOnSpot;
        require endurance;
        require wifiCommands;
        require wifiStatus;
        require safeStop;
        require cost;
        require massLimit;
        require chargingSafety;
    }

    requirement roverPresentationSpecification : PresentationRequirement {
        doc /* Requirements on how the rover is presented. */
        require scaleModelCaveat;
        require noPerformanceClaims;
    }
}


## 08_Verification_Traceability

In [ ]:
package LRD_Verification {
    doc /*
     * Verification cases: CK-1 to CK-5 and CK-H1 to CK-H8 from the specs, and the proposed
     * CK-6 to CK-9, CK-H9 and CK-H10 (PROPOSED in their names). Each check verifies only what
     * its pass condition tests. Also: satisfy relations, and allocation of run steps to parts.
     * None of the hardware checks has been run yet, because nothing is built.
     */

    private import VerificationCases::*;
    private import VerificationMethodKind::*;
    public import LRD_Requirements_Sim::*;
    public import LRD_Requirements_Hardware::*;
    public import LRD_Behaviour::*;

    // =====================================================================
    // Designs that the requirements are placed on
    // =====================================================================

    part mvpDesign : LunarRoadDemoSystem {
        part :>> rover {
            exhibit state modes : RoverModes;
        }
    }

    part roverBuild : TabletopRover;
    part demoPresentation : DemoPresentation;

    satisfy simulationSpecification by mvpDesign;
    satisfy presentationSpecification by demoPresentation;
    satisfy hardwareSpecification by roverBuild;
    satisfy roverPresentationSpecification by demoPresentation;

    // =====================================================================
    // Allocation: which logical element performs each step of a run
    // =====================================================================

    action runScenario : RunScenario;

    allocate runScenario.computeMaps to mvpDesign.environment;
    allocate runScenario.planRoute to mvpDesign.planner;
    allocate runScenario.delayCommand to mvpDesign.commLink;
    allocate runScenario.driveAndUpdatePower to mvpDesign.rover;
    allocate runScenario.logResults to mvpDesign.display;

    // =====================================================================
    // Verification cases
    // =====================================================================

    verification def 'CK-1 FairAndRepeatable' {
        doc /* Pass: Both runs differ only by the road setting, and the same seed gives the same
             * result. */
        @VerificationMethod { kind = test; }
        subject sim : LunarRoadDemoSystem;
        objective {
            verify roadComparison;
            verify fixedScenario;
            verify roadOnlyChangesFlags;
            verify repeatableRuns;
        }
    }

    verification def 'CK-2 LimitsRespected' {
        doc /* Pass: No route enters a cell above 15° or drives without Earth in view. */
        @VerificationMethod { kind = (test, analyze); }
        subject sim : LunarRoadDemoSystem;
        objective {
            verify slopeLimit;
            verify earthLimit;
            verify earthBeforeWaypoint;
        }
    }

    verification def 'CK-3 PowerModelCorrect' {
        doc /* Pass: With the Sun at 0° on a rear corner the solar input is 450 W, and a 600 W load
             * takes a full battery to empty in 9.0 h, both within 5% (TBR); with the Sun on a full
             * battery, state of charge stays at 1 and the surplus is discarded. The drain check
             * tests code only. */
        @VerificationMethod { kind = (analyze, test); }
        subject sim : LunarRoadDemoSystem;
        objective {
            verify solarReferenceCase;
            verify drainReferenceCase;
            verify chargeBounds;
            verify excessSolarDiscarded;
        }
    }

    verification def 'CK-4 ShadowsBelievable' {
        doc /* Pass: Shadows come from terrain, the shadow calculation uses the wider terrain map,
             * and at least 80% (TBR) of shadow-edge speeds fall within the published 0.1 to 1.8
             * cm/s. */
        @VerificationMethod { kind = analyze; }
        subject sim : LunarRoadDemoSystem;
        objective {
            verify visibilityFromTerrain;
            verify widerTerrainForShadows;
            verify shadowCredibility;
        }
    }

    verification def 'CK-5 DemoWorksEndToEnd' {
        doc /* Pass: A viewer can change every control and see the route, charge, shadow and results
             * update from the stored run set. */
        @VerificationMethod { kind = demo; }
        subject sim : LunarRoadDemoSystem;
        objective {
            verify precomputedRunSet;
        }
    }

    verification def 'CK-6 PROPOSED PlannerPowerAndDelayTests' {
        doc /* Proposed check, not in the original spec. Pass: Unit tests pass for: preset values on
             * road and off-road cells; heading choice in sun and shadow; the reserve limit; time and
             * energy; route cost; plan time and full-resolution recheck; delay sampling, status
             * period, command acknowledgement and resend, no status resend, and logging; failure
             * conditions. */
        @VerificationMethod { kind = test; }
        subject sim : LunarRoadDemoSystem;
        objective {
            verify offRoadMobility;
            verify roadMobility;
            verify headingOffsetLimit;
            verify sunlitHeading;
            verify shadowHeading;
            verify timeAndEnergy;
            verify reserveLimit;
            verify routeCost;
            verify planTime;
            verify fullResolutionRecheck;
            verify delaySampling;
            verify statusPeriod;
            verify commandBeforeWaypoint;
            verify commandResend;
            verify statusReplacement;
            verify dropAndResendLogging;
            verify failureConditions;
            verify commandAcknowledgement;
        }
    }

    verification def 'CK-7 PROPOSED PresentationReview' {
        doc /* Proposed check, not in the original spec. Pass: Every shared result and presentation
             * reports results either way, carries both caveats, and describes R2 as selected. */
        @VerificationMethod { kind = inspect; }
        subject presentation : DemoPresentation;
        objective {
            verify reportEitherWay;
            verify caveatSimplifiedModel;
            verify caveatAutonomy;
            verify r2Framing;
        }
    }

    verification def 'CK-8 PROPOSED OutputsAndConfigReview' {
        doc /* Proposed check, not in the original spec. Pass: Run outputs contain every reported
             * result, the risk and reliability measures, and the road-on minus road-off differences;
             * the scenario file exposes every value marked Assumed; the site and patch follow the
             * selection rules; the run set is generated and stored without the final display
             * installed. */
        @VerificationMethod { kind = inspect; }
        subject sim : LunarRoadDemoSystem;
        objective {
            verify reportedResults;
            verify riskAndReliability;
            verify reliabilityMeasure;
            verify comparisonDifference;
            verify assumedValuesConfigurable;
            verify demoSite;
            verify patchSelectionRules;
            verify displayIndependence;
        }
    }

    verification def 'CK-9 PROPOSED InterfaceInspection' {
        doc /* Proposed check, not in the original spec. Pass: Each interface carries the listed
             * content in the listed format between the listed blocks. */
        @VerificationMethod { kind = inspect; }
        subject sim : LunarRoadDemoSystem;
        objective {
            verify terrainInterface;
            verify ephemerisInterface;
            verify environmentMapsInterface;
            verify commandInterface;
            verify statusInterface;
            verify runLogInterface;
            verify scenarioInterface;
        }
    }

    verification def 'CK-H1 MeasureFinishedRover' {
        doc /* Pass: 170 × 170 mm footprint and 50 × 20 mm wheels, within ±2 mm; all listed features
             * present. */
        @VerificationMethod { kind = inspect; }
        subject rover : TabletopRover;
        objective {
            verify footprint;
            verify wheelSize;
            verify boxBody;
            verify cornerWheels;
            verify sidePanels;
            verify frontMast;
        }
    }

    verification def 'CK-H2 PrintEveryPart' {
        doc /* Pass: Each part prints on a 180 mm bed without supports. */
        @VerificationMethod { kind = inspect; }
        subject rover : TabletopRover;
        objective {
            verify printable;
        }
    }

    verification def 'CK-H3 LevelSpeedRun' {
        doc /* Pass: Driving 1 m at a commanded 2 cm/s on level ground takes 45.5 to 55.5 s. */
        @VerificationMethod { kind = test; }
        subject rover : TabletopRover;
        objective {
            verify levelSpeed;
        }
    }

    verification def 'CK-H4 RampAndTurn' {
        doc /* Pass: Drives up a 15° ramp of the demo surface without stalling, then completes a 360°
             * turn on the flat demo surface. */
        @VerificationMethod { kind = test; }
        subject rover : TabletopRover;
        objective {
            verify slopeClimb;
            verify turnOnSpot;
        }
    }

    verification def 'CK-H5 EnduranceRun' {
        doc /* Pass: Driving continuously from freshly charged cells, still driving after 2 h. */
        @VerificationMethod { kind = test; }
        subject rover : TabletopRover;
        objective {
            verify endurance;
        }
    }

    verification def 'CK-H6 WaypointsAndCommsLoss' {
        doc /* Pass: Five waypoints sent over Wi-Fi are followed, and status messages with position,
             * heading and battery voltage return; after sending stops, the rover stops within 1 s
             * once 1 s has passed with no message. */
        @VerificationMethod { kind = test; }
        subject rover : TabletopRover;
        objective {
            verify wifiCommands;
            verify wifiStatus;
            verify safeStop;
        }
    }

    verification def 'CK-H7 TotalReceipts' {
        doc /* Pass: Total under £300. */
        @VerificationMethod { kind = inspect; }
        subject rover : TabletopRover;
        objective {
            verify cost;
        }
    }

    verification def 'CK-H8 WeighRover' {
        doc /* Pass: Under 0.8 kg. */
        @VerificationMethod { kind = inspect; }
        subject rover : TabletopRover;
        objective {
            verify massLimit;
        }
    }

    verification def 'CK-H9 PROPOSED CaveatCheck' {
        doc /* Proposed check, not in the original spec. Pass: Everything shown with the rover
             * carries both caveats. */
        @VerificationMethod { kind = inspect; }
        subject presentation : DemoPresentation;
        objective {
            verify scaleModelCaveat;
            verify noPerformanceClaims;
        }
    }

    verification def 'CK-H10 PROPOSED ChargingCheck' {
        doc /* Proposed check, not in the original spec. Pass: Inspection shows no charge path to the
             * fitted cells. */
        @VerificationMethod { kind = inspect; }
        subject rover : TabletopRover;
        objective {
            verify chargingSafety;
        }
    }

}


## Diagrams

### System context (IF-01, IF-02)

In [ ]:
%viz --view=INTERCONNECTION LRD_Structure::demoContext

### Logical architecture with IF-03 to IF-07

In [ ]:
%viz --view=INTERCONNECTION LRD_Structure::LunarRoadDemoSystem

### Part tree

In [ ]:
%viz --view=TREE LRD_Structure::LunarRoadDemoSystem

### One simulation run

In [ ]:
%viz --view=ACTION LRD_Behaviour::RunScenario

### Rover operating modes

In [ ]:
%viz --view=STATE LRD_Behaviour::RoverModes

### Road preset variants

In [ ]:
%viz --view=TREE LRD_Variants::RoadPresetChoices

### 1:10 rover electrical design (E1 to E6)

In [ ]:
%viz --view=INTERCONNECTION LRD_Hardware::TabletopRover

### VIPER block to tabletop mapping

In [ ]:
%viz --view=TREE LRD_Hardware::viperReference LRD_Hardware::tabletopRoverDesign